In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · From a load test to Cloud Run settings

**What you will learn.** You will learn how the feedback loop of an overload looks in a load test, and what
admission control does to it. You will also learn how to change measurements into Cloud Run settings (concurrency,
min/max instances, billing mode) and into the "what breaks first" answer. The code is in `scalelab/sim.py`.

> **The one-minute version.** *"Before launch, I will use a load test that reproduces the incident mix to make
> sure that the design is correct. The load test gives me the turn duration. With Little's law, the turn duration gives me the in-flight
> turns, and the in-flight turns give the size of the orchestrator. The load test also tells me the cap above which a
> shed request is better for the user than a wait in a queue."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 140)
CLOCK.reset(0.02)   # 50x faster than real time
from scalelab.sim import resolve_backend
BACKEND, PROVIDER = resolve_backend()   # SCALELAB_BACKEND / SCALELAB_PROVIDER; unset: the hosted Gemini pool
print(f"backend: {BACKEND} ({PROVIDER})")

## 1. Four regimes

Each regime runs the same 120 users against a simulated 3 M TPM pool, with a different control:

- naive: retries only,
- degrade levels and a breaker, without a cap,
- degrade levels with an in-flight cap of 30.

Each run is approximately 90 virtual seconds (a few wall-clock seconds).

> **Which backend.** This notebook is for the default: a hosted pool that answers 429 (`SCALELAB_BACKEND`
> unset). If you start Jupyter with `SCALELAB_BACKEND=local` (or `hybrid`), the same load test runs on two simulated
> vLLM replicas instead. Then you get no 429s, the fleet holds the calls in a queue, and every turn becomes slower.
> Notebook 05 §5 puts the three backends side by side.
>
> Every cell still runs. The prose describes the hosted pool, and exercise (d) changes to the target of the fleet.

In [ ]:
from scalelab.sim import make_setup, simulate, compare
results = {}
for name, kw, users in [("moderate (20 users)", dict(pool_tpm=3_000_000), 20),
                        ("overload, naive", dict(pool_tpm=3_000_000, naive=True), 120),
                        ("overload, degrade only", dict(pool_tpm=3_000_000, max_inflight=1000), 120),
                        ("overload, degrade + cap 30", dict(pool_tpm=3_000_000, max_inflight=30), 120)]:
    CLOCK.reset(0.02)
    results[name] = await simulate(make_setup(**kw), users=users, duration_s=90, think_s=5)
summary = compare(results).round(3)
summary

In [ ]:
latency_cdfs(results)

In [ ]:
timeline(results["overload, naive"], "naive: the pool saturates, retries pile up, latency explodes" if BACKEND == "hosted"
         else f"naive on the {BACKEND} fleet: no 429s, the batch grows, latency explodes")

In [ ]:
timeline(results["overload, degrade + cap 30"], "cap 30: in-flight bounded, level rises, excess shed")

Read the table with the words of the primer:

- The naive run *is* the feedback loop. 429s cause retries, the retries make the turns long, and the long turns put
  more turns in flight.
- "Degrade only" changes the loop into breaker trips and hard failures.
- The cap changes the loop into shed requests with a `Retry-After`, and keeps every admitted turn fast. It does this
  at a fifth of the cost per turn, because most turns run on the lite tier.

## 2. From measurements to Cloud Run settings

Take the turn duration from the moderate run. Apply Little's law at the planned peak. Add headroom. Then divide by
the concurrency that one orchestrator instance can hold. The turns are I/O-bound coroutines, each with approximately
a few MB of context.

In [ ]:
from scalelab.capacity import Scenario, plan
s, p = Scenario(), plan(Scenario())
measured = results["moderate (20 users)"].summary()
turn_p50, turn_p95 = measured["p50_s"], measured["p95_s"]
for level in ("average", "peak", "incident"):
    turns_s = p["rates"][level]["turns_per_s"]
    inflight = turns_s * turn_p95                       # size for the tail, not the median
    print(f"{level:9s} {turns_s:5.1f} turns/s × {turn_p95:.1f} s = {inflight:6.0f} in flight → "
          f"{np.ceil(inflight * 1.4 / 80):3.0f} orchestrator instances at concurrency 80 (×1.4 headroom)")

| Setting | Gateway | Orchestrator | Why |
|---|---|---|---|
| Billing | request-based | **instance-based** | Checkpoints, compaction and telemetry run after the response. |
| Concurrency | 250 (SSE streams are low-cost) | 80 (memory per in-flight turn) | Cloud Run limits: 1,000 maximum, 800 req/s per instance. |
| Min / max instances | 2 / 100 | 2 / 50 | The min gives warm capacity. The regional quota and Direct VPC egress (~100–200/revision) set the limit of the max. |
| Timeout | 600 s | 600 s (= Pub/Sub ack deadline) | SSE streams are requests. The hard ceiling is 60 min. |
| Autoscaling | 60 % CPU or 60 % of max concurrency | same | Scale-out waits max(10 s, 3.5× predicted cold start). |

Cloud Run pricing (Tier 1, Sep 2026):

- request-based: \$0.000024/vCPU-s + \$0.0000025/GiB-s + \$0.40/M requests,
- instance-based: \$0.000018/vCPU-s + \$0.000002/GiB-s.

The whole fleet costs about 0.1 % of the model bill.

## 3. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn — solutions

#### (a) Instances needed

Return this value:

$$
\left\lceil \frac{\text{in-flight} \times \text{headroom}}{\text{concurrency}} \right\rceil,
$$

but never a value below `minimum`.

In [ ]:
def instances_needed(inflight, concurrency=80, headroom=1.4, minimum=2):
    return max(minimum, int(np.ceil(inflight * headroom / concurrency)))

In [ ]:
def _a():
    assert instances_needed(125) == 3 and instances_needed(417) == 8 and instances_needed(10) == 2
check("a: instances", _a)

#### (b) Cap from the token budget

Calculate how many turns can be in flight before they use all of a TPM budget:

$$
\frac{\mathrm{tpm}/60}{\text{tokens per turn} / \text{turn seconds}}.
$$

In [ ]:
def cap_from_budget(tpm, tokens_per_turn, turn_seconds):
    return (tpm / 60) / (tokens_per_turn / turn_seconds)

In [ ]:
def _b():
    assert abs(cap_from_budget(10_000_000, 2.2 * 5350, 6.0) - p["concurrency"]["max_inflight_for_baseline"]) < 0.01
    assert 20 < cap_from_budget(3_000_000, 2.5 * 3_800, 4.5) < 30
check("b: cap from budget", _b)

#### (c) Cloud Run monthly cost

Calculate the monthly cost from the vCPU-seconds and GiB-seconds for `instances` that run all month (730 h), in either billing mode. Request-based billing adds $0.40 per million requests.

In [ ]:
def cloud_run_monthly_usd(vcpu, gib, instances, mode, requests_per_month=0):
    seconds = 730 * 3600 * instances
    if mode == "instance":
        return seconds * (vcpu * 0.000018 + gib * 0.000002)
    return seconds * (vcpu * 0.000024 + gib * 0.0000025) + requests_per_month / 1e6 * 0.40

In [ ]:
def _c():
    assert abs(cloud_run_monthly_usd(2, 2, 3, "instance") - 730*3600*3*(2*0.000018 + 2*0.000002)) < 1e-6
    assert cloud_run_monthly_usd(1, 0.5, 2, "request", 18_000_000) > cloud_run_monthly_usd(1, 0.5, 2, "instance")
    print(f"   orchestrator fleet (3 × 2 vCPU / 2 GiB, instance-based): ${cloud_run_monthly_usd(2, 2, 3, 'instance'):,.0f}/month vs model bill ${p['cost']['per_month_routed_cached']:,.0f}")
check("c: Cloud Run cost", _c)

#### (d) Choose the cap

Select an in-flight cap for the 3 M TPM pool with 120 users. The cap must give p95 ≤ 8 s and no failed turn. Keep the shed rate under 30 %. The check runs a simulation with your value.

If you set `SCALELAB_BACKEND=local` or `hybrid`, the check runs two vLLM replicas instead. It then uses the fleet
target of notebook 05: p95 ≤ 7.5 s and a shed rate under 45 %. The reason is that on the fleet, the p95 is already
about 8 s when the shed rate decreases to under 30 % (simulated). Thus no cap meets the target of the pool with any
margin.

In [ ]:
MY_CAP = 40 if BACKEND == "hosted" else 30   # a fleet needs a tighter cap (notebook 05, exercise f)

In [ ]:
P95_MAX, SHED_MAX = (8.0, 0.30) if BACKEND == "hosted" else (7.5, 0.45)   # the pool's target; a fleet's (notebook 05 f)
async def _d():
    if MY_CAP is None:
        todo()
    CLOCK.reset(0.02)
    r = await simulate(make_setup(pool_tpm=3_000_000, max_inflight=MY_CAP), users=120, duration_s=60, think_s=5)
    s = r.summary()
    print(f"   cap {MY_CAP} on {BACKEND}: p95 {s['p95_s']:.1f} s (target ≤ {P95_MAX:g}), shed {s['shed_rate']:.0%} "
          f"(target < {SHED_MAX:.0%}), failed {s['failed']}, 429s {s['rate_limited_calls']}")
    assert s["p95_s"] <= P95_MAX and s["failed"] == 0 and s["shed_rate"] < SHED_MAX
await acheck("d: choose the cap", _d)

## Takeaways

- A load test gives you the turn duration. Little's law gives you the in-flight turns. Headroom and the concurrency per instance give you the instances.
- The naive system does not fail. It becomes slow for everyone. Admission control changes that into bounded latency and honest `Retry-After`s.
- Instance-based billing goes on the orchestrator (it does background work after the response). Request-based billing goes on the gateway.
- Say what breaks first: the token budget, then the slowest downstream system. Cloud Run is not on the list.

## Verify before relying on it

Examine these facts:

- The Cloud Run limits: concurrency 1,000, 800 req/s per instance, the 60-minute request ceiling, and the instance quota of Direct VPC egress.
- The pricing.
- The Preview status of custom autoscaling targets and Cloud Run *instances*.